In [1]:
import numpy as np
import openmdao.api as om
import os

os.environ['OPENMDAO_REPORTS'] = 'none'


class aerodynamicsDis(om.ExplicitComponent):
    def setup(self):
        # Global Design Variable
        self.add_input('B', val=1.)

        # Coupling parameter
        self.add_input('phi', val=0.)

        # Coupling output
        self.add_output('L', val=0.)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        q = 1 # N/cm2
        C = 10 # cm
        psi = 0.05 # rad
        r = 0.9425
        theta0 = 0.26 # rad

        B = inputs['B']
        phi = inputs['phi']

        outputs['L'] = 1/1000 * q*B*C * ((2*np.pi*(phi+psi)) + r*(1-np.cos(np.pi/2*(phi+psi)/theta0)))

class structuresDis(om.ExplicitComponent):
    def setup(self):
        # Coupling parameter
        self.add_input('L', val=0.)

        # Coupling output
        self.add_output('phi', val=0.)

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        C = 10 # cm
        p = 0.1111
        k1 = 4000 # N/cm
        k2 = 2000 # N/cm
        z1 = 0.2
        z2 = 0.7

        L = inputs['L']

        outputs['phi'] = np.remainder(( 1000*L/(k1*(1+p)) - (1000*L*p)/(k2*(1+p)) ) * ( 1/(C*(z2-z1)) ),2*np.pi)
        
class aerostructuresGroup(om.Group):
    def setup(self):
        cycle = self.add_subsystem('cycle', om.Group(), promotes=['*'])
        cycle.add_subsystem('aero', aerodynamicsDis(), promotes_inputs=['B', 'phi'],
                            promotes_outputs=['L'])
        cycle.add_subsystem('strux', structuresDis(), promotes_inputs=['L'],
                            promotes_outputs=['phi'])

        cycle.linear_solver = om.DirectSolver(rhs_checking=False)
        nlbgs = cycle.nonlinear_solver = om.NonlinearBlockGS()
        nlbgs.options['maxiter'] = 1000
        nlbgs.options['iprint'] = 2
        

In [4]:
prob = om.Problem()
prob.model = aerostructuresGroup()

prob.driver = om.ScipyOptimizeDriver()
prob.driver.options['optimizer'] = 'COBYQA'
prob.driver.options['tol'] = 1e-8
prob.driver.options['disp'] = False

prob.model.add_design_var('B', lower = 0., upper = 300.)

prob.model.approx_totals()

prob.setup()

prob.set_val('B', 300.)

prob.run_model()

print('coupling vars')
print('L', prob.get_val('L'))
print('phi', prob.get_val('phi'))

print('iter count', 
      prob.model.cycle.aero.iter_count + prob.model.cycle.strux.iter_count)


=====
cycle
=====
NL: NLBGS 1 ; 1.07116008 1
NL: NLBGS 2 ; 0.964535212 0.900458512
NL: NLBGS 3 ; 0.975705093 0.910886348
NL: NLBGS 4 ; 1.07557352 1.00412025
NL: NLBGS 5 ; 1.26693422 1.18276833
NL: NLBGS 6 ; 1.56352915 1.45965965
NL: NLBGS 7 ; 1.96178922 1.83146222
NL: NLBGS 8 ; 2.36675976 2.20952947
NL: NLBGS 9 ; 2.47746475 2.31288003
NL: NLBGS 10 ; 1.92704085 1.79902228
NL: NLBGS 11 ; 0.972455344 0.907852488
NL: NLBGS 12 ; 0.32803893 0.306246411
NL: NLBGS 13 ; 0.0888854029 0.0829805036
NL: NLBGS 14 ; 0.0223336705 0.020849984
NL: NLBGS 15 ; 0.00549755361 0.00513233616
NL: NLBGS 16 ; 0.00134627984 0.0012568428
NL: NLBGS 17 ; 0.000329267477 0.000307393342
NL: NLBGS 18 ; 8.05057785e-05 7.51575605e-05
NL: NLBGS 19 ; 1.96821341e-05 1.83745964e-05
NL: NLBGS 20 ; 4.81181841e-06 4.49215622e-06
NL: NLBGS 21 ; 1.17637096e-06 1.09822144e-06
NL: NLBGS 22 ; 2.87593371e-07 2.6848776e-07
NL: NLBGS 23 ; 7.03093917e-08 6.56385474e-08
NL: NLBGS 24 ; 1.71888883e-08 1.60469836e-08
NL: NLBGS 25 ; 4.202252